# Task 1: Load and Inspect the Data

In [82]:
# importing required libraries for this assignment
import pandas as pd
import numpy as np

 Loaded the dataset

In [3]:
data = {
    'transaction_id': range(1, 21),
    'date': pd.date_range('2024-10-01', periods=20, freq='D'),
    'region': ['North', 'South', 'East', 'West', None, 'North', 'South', None, 'East', 'West',
               'North', 'South', 'East', 'West', 'North', None, 'East', 'West', 'North', 'South'],
    'product_category': ['Electronics', 'Clothing', None, 'Books', 'Electronics', 'Home', 
                         'Clothing', 'Books', 'Electronics', None, 'Home', 'Clothing', 
                         'Books', 'Electronics', 'Home', 'Clothing', 'Books', 'Electronics', 
                         'Home', 'Clothing'],
    'sales_amount': [1200, 450, 890, None, 1500, 670, None, 340, 2100, 780, 
                     560, None, 420, 1800, 920, 510, 380, None, 1100, 640],
    'quantity': [2, 5, 3, 1, None, 4, 2, 3, 1, 5, 
                 3, 2, None, 1, 4, 3, 2, 1, None, 4],
    'customer_age': [25, 34, None, 45, 29, None, 38, 52, 27, 41, 
                     33, None, 48, 26, 35, 42, None, 31, 39, 44],
    'payment_method': ['Credit Card', 'UPI', 'Cash', 'Debit Card', 'Credit Card', 
                       'UPI', 'Cash', None, 'Credit Card', 'UPI', 'Debit Card', 
                       'Cash', 'Credit Card', None, 'UPI', 'Cash', 'Debit Card', 
                       'Credit Card', 'UPI', None]
}

df = pd.DataFrame(data)

1. Basic information like rows, dtypes, count of current dataset is provided when we use df.info()

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    20 non-null     int64         
 1   date              20 non-null     datetime64[ns]
 2   region            17 non-null     object        
 3   product_category  18 non-null     object        
 4   sales_amount      16 non-null     float64       
 5   quantity          17 non-null     float64       
 6   customer_age      16 non-null     float64       
 7   payment_method    17 non-null     object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(3)
memory usage: 1.4+ KB


2. we can count no.of missing values per each column using df.isna().sum() 

In [7]:
df.isna().sum()

transaction_id      0
date                0
region              3
product_category    2
sales_amount        4
quantity            3
customer_age        4
payment_method      3
dtype: int64

# Task 2: Handle Missing Values

1. For 'region' and 'product_category', to replace missing values we used mode because as they are categorical columns

In [11]:
df['region'] = df['region'].fillna(df['region'].mode()[0])

In [17]:
df['product_category'] = df['product_category'].fillna(df['product_category'].mode()[0])

2. Used Median to replace missing values in sales_amount as it may contain outliers to use mean and median is very efficient in these cases (using median())

In [25]:
df['sales_amount'] = df['sales_amount'].fillna(df['sales_amount'].median())

3. For 'quantity' column, we used ffill method in fillna because the data is time series or ordered

In [31]:
df['quantity'] = df['quantity'].fillna(method = 'ffill')

C:\Users\rahit\AppData\Local\Temp\ipykernel_13744\1201256151.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['quantity'] = df['quantity'].fillna(method = 'ffill')


4. Used mean to replace missing values in 'age' column as age can be filled with average cases

In [35]:
df['customer_age'] = df['customer_age'].fillna(df['customer_age'].mean())

5. For 'payment_method', to delete rows where it is missed we used subset and inplace true to delete in actual dataset 

In [39]:
df.dropna(subset = ['payment_method'], inplace = True)

checking to ensure no missing values in any column after cleaning

In [41]:
df.isna().sum()

transaction_id      0
date                0
region              0
product_category    0
sales_amount        0
quantity            0
customer_age        0
payment_method      0
dtype: int64

# Task 3: GroupBy Analysis

1. Calculating total sales by region using groupby method and them sum

In [45]:
df.groupby('region')['sales_amount'].sum()

region
East     3790.0
North    6460.0
South    1900.0
West     2230.0
Name: sales_amount, dtype: float64

2. Calculating average_sales in each product_category using groupby and then mean

In [47]:
df.groupby('product_category')['sales_amount'].mean()

product_category
Books           508.333333
Clothing        680.000000
Electronics    1381.250000
Home            812.500000
Name: sales_amount, dtype: float64

3. to find total sales and quantity in each region and under each category we used groupby for multiple columns and then sum for columns like sales_amount, quantity

In [59]:
df.groupby(['region','product_category'])[['sales_amount','quantity']].sum().reset_index()

,region,product_category,sales_amount,quantity
0,East,Books,800.0,4.0
1,East,Clothing,890.0,3.0
2,East,Electronics,2100.0,1.0
3,North,Clothing,510.0,3.0
4,North,Electronics,2700.0,3.0
5,North,Home,3250.0,12.0
6,South,Clothing,1900.0,9.0
7,West,Books,725.0,1.0
8,West,Clothing,780.0,5.0
9,West,Electronics,725.0,1.0


4. displayed top 3 region - product combinations by sales using groupby and sum, sort_values

In [63]:
res = df.groupby(['region','product_category'])['sales_amount'].sum().reset_index()
res.sort_values(by = 'sales_amount', ascending = False).head(3)

,region,product_category,sales_amount
5,North,Home,3250.0
4,North,Electronics,2700.0
2,East,Electronics,2100.0


# Task 4: Custom Aggregation

1. created a user defined function 'sales_range()' to return max-min
2. and then applied it to sales_amount column to find sales range in each region

In [73]:
def sales_range(n):
    return n.max() - n.min()
region_range = df.groupby('region')['sales_amount'].apply(sales_range)
region_range

region
East     1720.0
North     990.0
South     275.0
West       55.0
Name: sales_amount, dtype: float64

3. used .agg() to calculate multiple metrics by each region

In [79]:
# 3. Use .agg() for multiple metrics per region
region_summary = df.groupby('region').agg({
    'sales_amount': ['sum', 'mean', 'max', sales_range],
    'quantity': ['sum', 'min']
})

print(region_summary)

       sales_amount                                 quantity     
                sum        mean     max sales_range      sum  min
region                                                           
East         3790.0  947.500000  2100.0      1720.0      8.0  1.0
North        6460.0  922.857143  1500.0       990.0     18.0  1.0
South        1900.0  633.333333   725.0       275.0      9.0  2.0
West         2230.0  743.333333   780.0        55.0      7.0  1.0
